# Match BigEarthNet.txt to the existing 1,000 image pairs
Run cells one at a time. CPU is sufficient. This notebook downloads the official 467 MB text-only table once to the SatQuery source cache and matches records to the existing image metadata. It does not download the imagery again or train a model.

Both Sentinel IDs are checked. Source text is preserved. Original text splits and image splits remain separate; a benchmark annotation holds out its entire image. Missing images and conflicts are reported. Annotations describe the original 1.2 km area or explicit referenced regions; they must not be copied onto every 80 m token as if independently labeled.


## Install the checksum-verified SatQuery package

In [ ]:
import base64, hashlib, subprocess, sys
from pathlib import Path
wheel_path = Path('/content/satquery_preprocessing-0.1.0-py3-none-any.whl')
wheel_bytes = base64.b64decode(
    '...'
)
assert hashlib.sha256(wheel_bytes).hexdigest() == '9e37fa2b5aac68913d427e9b20aa165f818a815627bb83621c2ba7d504c890f3'
wheel_path.write_bytes(wheel_bytes)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', '--force-reinstall', '--no-deps', str(wheel_path)])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', str(wheel_path)+'[downloads,features]'])
print('Verified SatQuery package installed')


## Connect the existing Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Download and verify the official text source

In [ ]:
# Fetch the official text-only source; no satellite imagery is downloaded again.
from pathlib import Path
import pyarrow.parquet as pq
from satquery.download_bigearthnet import download_file
REVISION='72d865f2146f0a85b720f7f3ca1cdbaeafc3d316'
SOURCE_SHA256='d3b97f999456016bb13c2a8e94b8f47825654f07a0394a6b266a38b750ca1554'
SOURCE_URL=f'https://huggingface.co/datasets/BIFOLD-BigEarthNetv2-0/BigEarthNet.txt/resolve/{REVISION}/BigEarthNet.txt.parquet'
TEXT_SOURCE=Path('/content/drive/MyDrive/SatQuery/annotation-source-cache')/REVISION/'BigEarthNet.txt.parquet'
download_file(SOURCE_URL,466819745,SOURCE_SHA256,TEXT_SOURCE,progress=lambda s:print(s,flush=True))
parquet=pq.ParquetFile(TEXT_SOURCE)
print('OFFICIAL TEXT SOURCE VERIFIED:',parquet.metadata.num_rows,'records',flush=True)
print(parquet.schema_arrow,flush=True)


## Match annotations and verify links to the saved features

In [ ]:
# Match the official text records to existing images, then link to saved features.
import json
from pathlib import Path
import pandas as pd
from satquery.match_annotations import match_annotations, SOURCE_REVISION, SOURCE_SHA256
from satquery.preprocessing import sha256
from satquery.prediction_data import checked_file

P=Path('/content/drive/MyDrive/SatQuery/pipeline-1000')
IMAGE_ROOT=P.parent/'bigearthnet-v2-1000'
TEXT_SOURCE=P.parent/'annotation-source-cache'/SOURCE_REVISION/'BigEarthNet.txt.parquet'
OUTPUT=P/'annotations'
download=json.loads((IMAGE_ROOT/'download.json').read_text())
assert sha256(IMAGE_ROOT/'metadata.parquet')==download['metadata_sha256']
metadata=pd.read_parquet(IMAGE_ROOT/'metadata.parquet')
assert len(metadata)==1000
if not OUTPUT.exists():
    match_annotations(IMAGE_ROOT/'metadata.parquet',TEXT_SOURCE,OUTPUT,
                      source_revision=SOURCE_REVISION,expected_source_sha256=SOURCE_SHA256,
                      progress=lambda s:print(s,flush=True))
report=json.loads((OUTPUT/'report.json').read_text())
assert report['source_sha256']==SOURCE_SHA256
assert report['selected_metadata_sha256']==download['metadata_sha256']
for name,digest in report['files'].items(): assert sha256(OUTPUT/name)==digest
features=P/'features'
fm=json.loads((features/'manifest.json').read_text())
previous=json.loads((P/'stage-2-report.json').read_text())
assert sha256(features/'manifest.json')==previous['feature_manifest_sha256']
scenes=[json.loads(line) for line in (OUTPUT/'image-annotations.jsonl').read_text().splitlines()]
assert [s['patch_id'] for s in scenes]==metadata.patch_id.tolist()
links=[]
for batch in fm['batches']:
    info=json.loads(checked_file(features.resolve(),batch,'batch.json'))
    for offset,sample in enumerate(info['samples']):
        index=batch['start_index']+offset
        scene=scenes[index]
        assert scene['image_index']==index
        assert scene['patch_id']==sample['patch_id'] and scene['s1_name']==sample['s1_name']
        assert scene['image_split']==sample['split']
        links.append({k:scene[k] for k in ['patch_id','s1_name','image_index','image_split','annotation_splits','annotation_count','use_partition']} |
                     {'feature_batch':batch['directory'],'index_in_feature_batch':offset,
                      'feature_file_sha256':batch['sha256']['features.pt'],'feature_manifest_sha256':previous['feature_manifest_sha256']})
assert len(links)==1000
linkfile=OUTPUT/'feature-links.jsonl'
text=''.join(json.dumps(row)+'\n' for row in links)
if linkfile.exists(): assert linkfile.read_text()==text
else: linkfile.write_text(text)
receipt={**report,'feature_manifest_sha256':previous['feature_manifest_sha256'],
         'feature_link_count':len(links),'feature_links_sha256':sha256(linkfile),
         'matched_report_sha256':sha256(OUTPUT/'report.json'),
         'matched_output_bytes':sum(p.stat().st_size for p in OUTPUT.iterdir() if p.is_file()),
         'output':str(OUTPUT)}
(P/'stage-8-annotations-report.json').write_text(json.dumps(receipt,indent=2)+'\n')
print('ANNOTATIONS MATCHED:',json.dumps(receipt),flush=True)
example=next(s for s in scenes if s['annotations'])
print('EXAMPLE:',json.dumps({'patch_id':example['patch_id'],'s1_name':example['s1_name'],
                           'image_split':example['image_split'],'annotation_count':example['annotation_count'],
                           'annotations':[{k:r[k] for k in ['ID','input','output','type','category','split']} for r in example['annotations'][:3]]}),flush=True)


## Outputs
Google Drive → MyDrive → SatQuery → pipeline-1000 → annotations:
- `annotations.parquet`: all matching source records plus image split, image index, permitted partition and training eligibility.
- `image-annotations.jsonl`: one entry per selected image pair, with its linked annotations; missing images remain visible.
- `feature-links.jsonl`: exact feature batch and position for every image.
- `report.json`: source hashes, record counts, missing IDs and split conflicts.

The parent folder contains `stage-8-annotations-report.json`, including verified feature links. No training occurs in this notebook.
